In [17]:
import kagglehub
import pandas as pd
import os

# Download latest version
path = kagglehub.dataset_download("saurabhshahane/fake-news-classification")

print("Path to dataset files:", path)

print(path)

Path to dataset files: C:\Users\pedro\.cache\kagglehub\datasets\saurabhshahane\fake-news-classification\versions\77
C:\Users\pedro\.cache\kagglehub\datasets\saurabhshahane\fake-news-classification\versions\77


In [18]:
os.listdir(path)

['WELFake_Dataset.csv']

In [19]:
df = pd.read_csv(os.path.join(path, "WELFake_Dataset.csv"))
df.head(10)

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1
5,5,About Time! Christian Group Sues Amazon and SP...,All we can say on this one is it s about time ...,1
6,6,DR BEN CARSON TARGETED BY THE IRS: “I never ha...,DR. BEN CARSON TELLS THE STORY OF WHAT HAPPENE...,1
7,7,HOUSE INTEL CHAIR On Trump-Russia Fake Story: ...,,1
8,8,Sports Bar Owner Bans NFL Games…Will Show Only...,"The owner of the Ringling Bar, located south o...",1
9,9,Latest Pipeline Leak Underscores Dangers Of Da...,"FILE – In this Sept. 15, 2005 file photo, the ...",1


In [20]:
print(df.isnull().sum())
df = df.dropna(subset=['title', 'text'])

Unnamed: 0      0
title         558
text           39
label           0
dtype: int64


In [21]:
import pandas as pd
import re
import spacy
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

# Carregar o spaCy desativando componentes pesados para ganhar velocidade
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def nlp_cleaner(text):
    # 1. Limpeza Bruta (RE)
    text = re.sub(r'http\S+|www\S+|https\S+', '', str(text), flags=re.MULTILINE)
    text = re.sub(r'[^a-zA-Z\s]', '', text).lower().strip()
    
    # 2. Inteligência Linguística (spaCy)
    doc = nlp(text)
    # Lemmatization: transforma 'running' em 'run', unificando o sinal
    return " ".join([token.lemma_ for token in doc if not token.is_stop and not token.is_punct])

# Criando a coluna de sinal máximo
df['total_content'] = df['title'] + " " + df['text']

# Processamento em lote (Batch) para não travar o kernel
print("Iniciando limpeza NLP...")
df['cleaned_text'] = [nlp_cleaner(t) for t in tqdm(df['total_content'])]

Iniciando limpeza NLP...


100%|██████████| 71537/71537 [28:29<00:00, 41.84it/s]  


In [22]:
# Divisão Treino/Teste
X_train, X_test, y_train, y_test = train_test_split(df['cleaned_text'], df['label'], test_size=0.2, random_state=42)

# TF-IDF com N-grams (1,2) para captar expressões compostas
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

# Modelo
pac = PassiveAggressiveClassifier(max_iter=50)
pac.fit(X_train_vec, y_train)

# Avaliação
y_pred = pac.predict(X_test_vec)
print(f'Acurácia: {accuracy_score(y_test, y_pred)*100:.2f}%')
print(classification_report(y_test, y_pred))

Acurácia: 95.17%
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      7081
           1       0.95      0.96      0.95      7227

    accuracy                           0.95     14308
   macro avg       0.95      0.95      0.95     14308
weighted avg       0.95      0.95      0.95     14308



In [23]:
import joblib

# Salvando o modelo e o vetorizador TF-IDF
joblib.dump(pac, 'modelo_fakenews_95.pkl')
joblib.dump(tfidf, 'vetorizador_tfidf.pkl')

print("Arquivos salvos com sucesso!")

Arquivos salvos com sucesso!


In [25]:
import pandas as pd
import joblib
import re
import spacy
import time

# 1. Setup inicial
print("🔍 Carregando Pipeline de NLP e Base WELFake...")
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
model = joblib.load('modelo_fakenews_95.pkl')
tfidf = joblib.load('vetorizador_tfidf.pkl')
df 

def nlp_cleaner(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', str(text), flags=re.MULTILINE)
    text = re.sub(r'[^a-zA-Z\s]', '', text).lower().strip()
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc if not token.is_stop]) #

def mostrar_noticia():
    # Sorteia uma notícia aleatória
    amostra = df.sample(1).iloc[0]
    
    # Processamento para predição
    sinal = str(amostra['title']) + " " + str(amostra['text'])
    limpo = nlp_cleaner(sinal)
    vetor = tfidf.transform([limpo])
    pred = model.predict(vetor)[0]
    
    # Mapeamento de Labels (WELFake: 1=Fake, 0=Real)
    label_real = "FAKE 🚩" if amostra['label'] == 1 else "REAL ✅"
    label_pred = "FAKE 🚩" if pred == 1 else "REAL ✅"
    cor = "\033[91m" if pred == 1 else "\033[92m"
    resultado = "🎯 ACERTO" if pred == amostra['label'] else "❌ ERRO"

    # --- FORMATO DE SAÍDA PARA O TERMINAL ---
    print("\n" + "═"*70)
    print(f"📌 TÍTULO: {amostra['title']}")
    print("-" * 70)
    # Exibe os primeiros 400 caracteres para dar contexto no slide
    print(f"📄 TEXTO: {amostra['text'][:400]}...") 
    print("-" * 70)
    
    print(f"📊 GABARITO (Dataset): {label_real}")
    print(f"🤖 IA PREDIZ: {cor}{label_pred}\033[0m")
    print(f"⚖️ VEREDITO: {resultado}")
    print("═"*70 + "\n")

if __name__ == "__main__":
    try:
        while True:
            mostrar_noticia()
            # O input garante que o terminal pare e espere você bater o print
            opcao = input("Pressione ENTER para próximo sorteio ou 'q' para sair: ").lower()
            if opcao == 'q':
                break
    except KeyboardInterrupt:
        print("\nSessão encerrada.")

🔍 Carregando Pipeline de NLP e Base WELFake...

══════════════════════════════════════════════════════════════════════
📌 TÍTULO: Health Secretary Price believes has president's confidence
----------------------------------------------------------------------
📄 TEXTO: WASHINGTON (Reuters) - U.S. Health and Human Services Secretary Tom Price said on Thursday he believed he still had President Trump’s confidence, a day after the president said he was not happy with Price over reports he had used private planes for official travel. “I think we’ve still got the confidence of the president,” Price told reporters who asked whether he expected to be fired over the mat...
----------------------------------------------------------------------
📊 GABARITO (Dataset): REAL ✅
🤖 IA PREDIZ: REAL ✅
⚖️ VEREDITO: 🎯 ACERTO
══════════════════════════════════════════════════════════════════════


══════════════════════════════════════════════════════════════════════
📌 TÍTULO: The best evidence yet that Repub